In [2]:
%load_ext sql

In [1]:
%sql spark

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/zorder_ex1 --recursive

In [2]:

df = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_201_99457.parquet")
df = df.select("customer_id", "category", "price", "quantity", "invoice_date")

In [3]:
display(df.limit(5))

+-----------+--------+------+--------+------------+
|customer_id|category|price |quantity|invoice_date|
+-----------+--------+------+--------+------------+
|201        |Clothing|900.24|3       |2021-07-04  |
|202        |Clothing|900.24|3       |2022-01-14  |
|203        |Toys    |35.84 |1       |2022-02-20  |
|204        |Clothing|1500.4|5       |2022-06-18  |
|205        |Souvenir|35.19 |3       |2022-04-27  |
+-----------+--------+------+--------+------------+



In [4]:
df_union = df
expected_rows = 20000000 # 20,000,000

while df_union.count() <= expected_rows:
    df_union = df_union.union(df_union)
    print(f"count: {df_union.count()}")

print(f"final count: {df_union.count()}")

count: 198514
count: 397028
count: 794056
count: 1588112
count: 3176224
count: 6352448
count: 12704896
count: 25409792
final count: 25409792


In [5]:

df_union.write.format("delta").mode("overwrite").saveAsTable("spark_catalog.deltalake.zorder_ex1")

In [6]:
%%time

spark.sql(
    """
    SELECT category,
        SUM(price * quantity) as total_sales
    FROM spark_catalog.deltalake.zorder_ex1
    WHERE customer_id = 201
    GROUP BY category
    """
)

CPU times: user 1.27 ms, sys: 1.99 ms, total: 3.26 ms
Wall time: 191 ms


DataFrame[category: string, total_sales: double]

In [7]:
%%sql

OPTIMIZE spark_catalog.deltalake.zorder_ex1
ZORDER BY customer_id;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
s3a://warehouse/default/deltalake/zorder_ex1,"Row(numFilesAdded=1, numFilesRemoved=256, filesAdded=Row(min=172182853, max=172182853, avg=172182853.0, totalFiles=1, totalSize=172182853), filesRemoved=Row(min=677463, max=677463, avg=677463.0, totalFiles=256, totalSize=173430528), partitionsOptimized=1, zOrderStats=Row(strategyName='all', inputCubeFiles=Row(num=0, size=0), inputOtherFiles=Row(num=256, size=173430528), inputNumCubes=0, mergedFiles=Row(num=256, size=173430528), numOutputCubes=1, mergedNumCubes=None), clusteringStats=None, numBins=1, numBatches=1, totalConsideredFiles=256, totalFilesSkipped=0, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1779512330745, endTimeMs=0, totalClusterParallelism=8, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=None, numTableColumns=5, numTableColumnsWithStats=5)"


In [0]:
# from delta.tables import DeltaTable

# table = DeltaTable.forName(spark, "spark_catalog.deltalake.zorder_ex1")
# table.optimize().executeZOrderBy("customer_id")

In [ ]:
# from delta.tables import DeltaTable

# delta_table = DeltaTable.forPath(spark, delta_path)
# delta_table.optimize().executeCompaction()

In [8]:
%%time

spark.sql(
    """
    SELECT category,
        SUM(price * quantity) as total_sales
    FROM spark_catalog.deltalake.zorder_ex1
    WHERE customer_id = 201
    GROUP BY category
    """
)

CPU times: user 1.46 ms, sys: 467 μs, total: 1.93 ms
Wall time: 53.4 ms


DataFrame[category: string, total_sales: double]

In [9]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/zorder_ex2 --recursive

In [10]:
#
df.write.format("delta").mode("overwrite").partitionBy("invoice_date").saveAsTable("spark_catalog.deltalake.zorder_ex2")

- Hive Style Partition: `invoice_date`
- ZORDER BY: `customer_id` 
- For each of those `invoice_date`

In [11]:
import pyspark.sql.functions as F

df_new_partition = df.filter(F.col("invoice_date") == "2023-01-01").withColumn("invoice_date", F.to_date(F.lit("2025-05-04")))


In [12]:

display(df_new_partition.limit(5))

+-----------+----------+-------+--------+------------+
|customer_id|category  |price  |quantity|invoice_date|
+-----------+----------+-------+--------+------------+
|986        |Clothing  |1500.4 |5       |2025-05-04  |
|1127       |Souvenir  |58.65  |5       |2025-05-04  |
|3271       |Shoes     |1800.51|3       |2025-05-04  |
|3485       |Technology|4200.0 |4       |2025-05-04  |
|4648       |Clothing  |1200.32|4       |2025-05-04  |
+-----------+----------+-------+--------+------------+



In [13]:

df_new_partition.write.format("delta").mode("append").partitionBy("invoice_date").saveAsTable("spark_catalog.deltalake.zorder_ex2")

In [14]:
%%sql

SELECT MAX(invoice_date) FROM spark_catalog.deltalake.zorder_ex2;

Running query in 'SparkSession'

1 rows affected.

Field 1
2025-05-04


In [15]:
%%sql

OPTIMIZE spark_catalog.deltalake.zorder_ex2 
WHERE invoice_date = '{current_day - 1}'
ZORDER BY customer_id;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
s3a://warehouse/default/deltalake/zorder_ex2,"Row(numFilesAdded=0, numFilesRemoved=0, filesAdded=Row(min=None, max=None, avg=0.0, totalFiles=0, totalSize=0), filesRemoved=Row(min=None, max=None, avg=0.0, totalFiles=0, totalSize=0), partitionsOptimized=0, zOrderStats=Row(strategyName='all', inputCubeFiles=Row(num=0, size=0), inputOtherFiles=Row(num=0, size=0), inputNumCubes=0, mergedFiles=Row(num=0, size=0), numOutputCubes=0, mergedNumCubes=None), clusteringStats=None, numBins=0, numBatches=1, totalConsideredFiles=0, totalFilesSkipped=0, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1779512428675, endTimeMs=0, totalClusterParallelism=8, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=None, numTableColumns=5, numTableColumnsWithStats=5)"
